In [2]:
%pip install phonenumbers --quiet

Note: you may need to restart the kernel to use updated packages.


In [8]:
import re
import phonenumbers

Xử lý cho poland

In [35]:
import pandas as pd

# Đường dẫn tới 2 file
gd_path = r"C:\Users\Nhung\Downloads\We_Love_Pho\sample structure.csv"
checkpoint_path = r"C:\Users\Nhung\Downloads\We_Love_Pho\raw_country_extracted\Poland.csv"

# 1. Đọc dữ liệu
df_checkpoint = pd.read_csv(checkpoint_path)
df_gd = pd.read_csv(gd_path)

In [36]:
df_checkpoint.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5102 entries, 0 to 5101
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   post_title          5102 non-null   object 
 1   address             5102 non-null   object 
 2   latitude            5102 non-null   float64
 3   longitude           5102 non-null   float64
 4   phone               4842 non-null   object 
 5   website             2899 non-null   object 
 6   facebook            0 non-null      float64
 7   instagram           0 non-null      float64
 8   twitter             0 non-null      float64
 9   post_content        5102 non-null   object 
 10  google_maps_link    5102 non-null   object 
 11  google_profile      5102 non-null   object 
 12  google_review_link  5102 non-null   object 
 13  country             5102 non-null   object 
dtypes: float64(5), object(9)
memory usage: 558.2+ KB


In [21]:
# --- Kiểm tra định dạng số điện thoại trước khi chuẩn hóa ---
phones = df_checkpoint["phone"].astype(str).str.strip()

# Loại bỏ các nan
valid_phones = phones[~phones.str.lower().isin(["nan", "none", ""]) & (phones != "")]

# Kiểm tra định dạng số điện thoại
# Có bắt đầu bằng '00'?
count_00 = valid_phones.str.startswith("00").sum()
# Có bắt đầu bằng '+' ?
count_plus = valid_phones.str.startswith("+").sum()
# Có dấu '-'?
count_dash = valid_phones.str.contains("-", regex=False).sum()
# Có dấu cách ?
count_space = valid_phones.str.contains(" ", regex=False).sum()

# Kiểm tra range số điện thoại
phones_cleaned = valid_phones.str.replace(r"[-\s]", "", regex=True)
lengths = phones_cleaned.str.len()
min_len = lengths.min()
max_len = lengths.max()

# --- In kết quả ---
print(f"Số bắt đầu bằng '00': {count_00}")
print(f"Số bắt đầu bằng '+': {count_plus}")
print(f"Số chứa dấu '-': {count_dash}")
print(f"Số chứa dấu cách: {count_space}")
print(f"Độ dài ngắn nhất (sau khi loại bỏ ký tự và NaN): {min_len}")
print(f"Độ dài dài nhất: {max_len}")


Số bắt đầu bằng '00': 0
Số bắt đầu bằng '+': 0
Số chứa dấu '-': 0
Số chứa dấu cách: 4842
Độ dài ngắn nhất (sau khi loại bỏ ký tự và NaN): 9
Độ dài dài nhất: 9


In [22]:
# Chuẩn hóa số điện thoại theo tiêu chuẩn E.164
from phonenumbers import PhoneNumberFormat
# ---- Country to Region Code Mapping ----
country_region_map = {
    "Sweden": "SE",
    "United Kingdom": "GB",
    "France": "FR",
    "Germany": "DE",
    "Poland": "PL",
    "Czech Republic": "CZ",
    "Slovakia": "SK",
    "Italy": "IT",
    "Spain": "ES",
    "Portugal": "PT",
    "Belgium": "BE",
    "Netherlands": "NL",
    "Hungary": "HU",
    "Austria": "AT"
}

# Hàm clean - giữ nan và xóa kí tự lạ (gồm dấu cách và - )
def clean_phone_number(raw_phone):
    if pd.isna(raw_phone):
        return raw_phone  
    raw_phone = str(raw_phone)
    cleaned = re.sub(r'[^\d+]', '', raw_phone)  
    return cleaned

# Chuẩn hóa số hợp lệ theo E.164 (thư viện phonenumbers để đưa về dạng sđt quốc tế)
def standardize_phone_number(row):
    raw = clean_phone_number(row["phone"])
    region = country_region_map.get(row["country"], None)
    if pd.isna(raw) or not str(raw).strip():
        return row["phone"]  
    try:
        parsed = phonenumbers.parse(raw, region)
        if phonenumbers.is_valid_number(parsed):
            return phonenumbers.format_number(parsed, PhoneNumberFormat.E164)
        else:
            return row["phone"]
    except:
        return row["phone"]

# Gán nhãn hợp lệ / không hợp lệ / thiếu để tiện lọc thủ công (nếu có)
def label_phone_status(row):
    raw = clean_phone_number(row["phone"])
    region = country_region_map.get(row["country"], None)
    if pd.isna(raw) or not str(raw).strip():
        return pd.NA 
    try:
        parsed = phonenumbers.parse(raw, region)
        if phonenumbers.is_valid_number(parsed):
            return 1  # Valid
        else:
            return 0  # Invalid
    except:
        return 0  # Invalid do lỗi

# Áp dụng hàm
df_checkpoint["phone"] = df_checkpoint.apply(standardize_phone_number, axis=1)
df_checkpoint["phone_status"] = df_checkpoint.apply(label_phone_status, axis=1)
print(df_checkpoint[["phone", "country", "phone_status"]].head(5))

          phone country phone_status
0  +48126388888  Poland            1
1  +48729297629  Poland            1
2  +48224681264  Poland            1
3  +48794456789  Poland            1
4  +48696145735  Poland            1


In [23]:
# Tách lấy street, zip và city từ address
# Xoá phần giống giá trị của country ở cuối address (có thể kèm số, dấu phẩy, khoảng trắng)
df_checkpoint['address'] = df_checkpoint.apply(
    lambda row: re.sub(rf'[,\s]*{re.escape(str(row["country"]))}\s*\d*$', '', str(row["address"]), flags=re.IGNORECASE).strip()
    if pd.notna(row["country"]) and row["country"] != "" else row["address"],
    axis=1
)

def split_address_with_zip_in_street(address):
    if pd.isna(address):
        return "", "", ""
    match = re.search(r'(\d{2,3}[-\s]?\d{3})\s*(.+)', address)
    if match:
        zip_code = match.group(1)
        city = match.group(2).strip()
        street = address[:match.end(1)].strip().rstrip(',')  # lấy từ đầu tới hết zip
        return street, zip_code, city
    return address, "", ""  # Nếu không match thì coi toàn bộ là street

# Áp dụng hàm cho từng dòng
df_checkpoint[['street', 'zip', 'city']] = df_checkpoint['address'].apply(
    lambda x: pd.Series(split_address_with_zip_in_street(x))
)
# xóa cột address
df_checkpoint.drop(columns=['address'], inplace=True)


In [24]:
# Kiểm tra active của web 
import requests
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor

# Chuẩn hóa URL
def normalize_url(url):
    if pd.isna(url) or not str(url).strip():
        return None
    url = url.strip()
    parsed = urlparse(url)
    if not parsed.scheme:
        return "http://" + url
    return url

# Kiểm tra hoạt động website, giữ original URL
def check_url(original_url):
    norm_url = normalize_url(original_url)
    if not norm_url:
        return (original_url, None, None, False)
    try:
        response = requests.get(norm_url, timeout=3, allow_redirects=True)
        final_url = response.url
        status = response.status_code
        is_active = 200 <= status < 400
        return (original_url, status, final_url, is_active)
    except:
        return (original_url, 0, None, False)

# Áp dụng đa luồng
df_checkpoint["normalized_url"] = df_checkpoint["website"].apply(normalize_url)
urls = df_checkpoint["normalized_url"].tolist()

with ThreadPoolExecutor(max_workers=30) as executor:
    results = list(executor.map(check_url, urls))

# Ghi kết quả vào DataFrame
df_checkpoint["web_status"] = [r[1] for r in results]
df_checkpoint["final_url"] = [r[2] for r in results]
df_checkpoint["is_active"] = [r[3] for r in results]


In [25]:
# Kiểm tra các link mạng xã hội bị lẫn trong website
# Xóa cột normalized_url
df_checkpoint.drop(columns=["normalized_url"], inplace=True)

# Xác định nền tảng mạng xã hội 
def classify_social_platform(url):
    if pd.isna(url):
        return None
    url = url.lower()
    if "facebook.com" in url:
        return "facebook"
    elif "instagram.com" in url:
        return "instagram"
    elif "twitter.com" in url or "x.com" in url:
        return "twitter"
    return None

df_checkpoint["social_platform"] = df_checkpoint["website"].apply(classify_social_platform)

# Chuyển các link sai về đúng cột
for platform in ["facebook", "instagram", "twitter"]:
    df_checkpoint[platform] = df_checkpoint.apply(
        lambda row: row["website"] if row["social_platform"] == platform and pd.isna(row[platform]) else row[platform],
        axis=1
    )

# Lưu kết quả vào file CSV
output_path = r"C:\Users\Nhung\Downloads\We_Love_Pho\Clean 25-5 - raw\poland_web_check.csv"

In [26]:
# Xoá khỏi giá trị cột website nếu là link MXH 
df_checkpoint.loc[df_checkpoint["social_platform"].notna(), "website"] = None
df_checkpoint.drop(columns=["social_platform"], inplace=True)

In [27]:
df_checkpoint.head(3)


,post_title,latitude,longitude,phone,website,facebook,instagram,twitter,post_content,google_maps_link,google_profile,google_review_link,country,phone_status,street,zip,city,web_status,final_url,is_active
0,"VIETNAM Food / Wietnamska kuchnia, kuchnia azj...",50.058408,19.946500,+48126388888,https://www.pyszne.pl/menu/viet-coffee-food%20...,NaN,NaN,NaN,"VIETNAM Food / Wietnamska kuchnia, kuchnia azj...",https://maps.google.com/?cid=15218949997944237800,https://maps.google.com/?q=place_id:ChIJM_VQvP...,https://search.google.com/local/reviews?placei...,Poland,1,"Wielopole 24/3, 31-072",31-072,Kraków,403.0,https://www.pyszne.pl/menu/viet-coffee-food%20...,False
1,Bar Asia Anh Viet-Thai Food,52.224834,21.015864,+48729297629,NaN,NaN,NaN,NaN,Bar Asia Anh Viet-Thai Food located in Wilcza ...,https://maps.google.com/?cid=10721585965558886871,https://maps.google.com/?q=place_id:ChIJ8Z0CMe...,https://search.google.com/local/reviews?placei...,Poland,1,"Wilcza 31, 00-538",00-538,Warszawa,NaN,None,False
2,Little Hanoi - Asian Fusion,52.232844,21.014736,+48224681264,None,https://www.facebook.com/littlehanoi.szpitalna3/,NaN,NaN,"Little Hanoi, Śródmieście, Warsaw. 3,141 likes...",https://maps.google.com/?cid=1568014939364155375,https://maps.google.com/?q=place_id:ChIJgfPpXf...,https://search.google.com/local/reviews?placei...,Poland,1,"Szpitalna 3, 00-031",00-031,Warszawa,200.0,https://www.facebook.com/littlehanoi.szpitalna3/,True


In [28]:
# tạo copy
df_checkpoint_copy = df_checkpoint.copy()

In [29]:
# 7. Lấy danh sách cột chuẩn từ file gd
gd_columns = df_gd.columns.tolist()

# 8. Thêm các cột còn thiếu và gán giá trị rỗng
for col in gd_columns:
    if col not in df_checkpoint_copy.columns:
        df_checkpoint_copy[col] = ""

# 9. Gán giá trị mặc định
df_checkpoint_copy['post_status'] = 'publish'
df_checkpoint_copy['post_category'] = ',249,'
df_checkpoint_copy['default_category'] = '249'
df_checkpoint_copy['featured'] = '0'

# 10. Sắp xếp lại đúng thứ tự cột
df_checkpoint_copy = df_checkpoint_copy[gd_columns]

# 12. Lưu file đã chuẩn hoá
df_checkpoint_copy.to_csv("poland_standardized.csv", index=False)

# 13. (Tuỳ chọn) Xem thử kết quả
print(df_checkpoint_copy[['street', 'zip', 'city', 'country']].head())

                    street     zip      city country
0   Wielopole 24/3, 31-072  31-072    Kraków  Poland
1        Wilcza 31, 00-538  00-538  Warszawa  Poland
2      Szpitalna 3, 00-031  00-031  Warszawa  Poland
3  Kolejowa 49A/U2, 01-210  01-210  Warszawa  Poland
4        Pańska 61, 00-830  00-830  Warszawa  Poland


In [31]:
df_checkpoint_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5102 entries, 0 to 5101
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                5102 non-null   object 
 1   post_title        5102 non-null   object 
 2   post_content      5102 non-null   object 
 3   post_status       5102 non-null   object 
 4   post_author       5102 non-null   object 
 5   post_type         5102 non-null   object 
 6   post_date         5102 non-null   object 
 7   post_modified     5102 non-null   object 
 8   post_tags         5102 non-null   object 
 9   post_category     5102 non-null   object 
 10  default_category  5102 non-null   object 
 11  featured          5102 non-null   object 
 12  street            5102 non-null   object 
 13  street2           5102 non-null   object 
 14  city              5102 non-null   object 
 15  region            5102 non-null   object 
 16  country           5102 non-null   object 


In [32]:
# in ra 5 giá trị đầu cột lat và long
print(df_checkpoint_copy[['latitude', 'longitude']].head())

    latitude  longitude
0  50.058408  19.946500
1  52.224834  21.015864
2  52.232844  21.014736
3  52.227640  20.983284
4  52.231808  20.995121


In [33]:
dups_full = df_checkpoint_copy[df_checkpoint_copy.duplicated(subset=['post_title', 'latitude', 'longitude', 'street'], keep=False)]
print(dups_full)

     ID                                         post_title  \
0        VIETNAM Food / Wietnamska kuchnia, kuchnia azj...   
1                              Bar Asia Anh Viet-Thai Food   
2                              Little Hanoi - Asian Fusion   
3                                            Hello Vietnam   
4                                          Hello Vietnam 2   
...  ..                                                ...   
5097                                          Pasta Miasta   
5098                                        Lolo Thai Jolo   
5099                    MyThai Restauracja Azjatycka Ustka   
5100     Restauracja Tuk Tuk (Bangkok) | Azjatycka rest...   
5101             Molo Beach Bar - Molo Surf Spot Jastarnia   

                                           post_content post_status  \
0     VIETNAM Food / Wietnamska kuchnia, kuchnia azj...     publish   
1     Bar Asia Anh Viet-Thai Food located in Wilcza ...     publish   
2     Little Hanoi, Śródmieście, Warsaw. 3

In [78]:
# Đếm số lượng giá trị không null cho từng dòng
df_checkpoint_copy['non_null_count'] = df_checkpoint_copy.notnull().sum(axis=1)

# Sắp xếp theo các cột và theo số lượng giá trị không null giảm dần
df_sorted = df_checkpoint_copy.sort_values(by=['post_title', 'latitude', 'longitude', 'street', 'non_null_count'], ascending=[True, True, True, True, False])

# Xóa các dòng trùng hoàn toàn, giữ lại dòng có nhiều thông tin nhất
df_deduplicated = df_sorted.drop_duplicates(keep='first').drop(columns=['non_null_count'])

# Lưu kết quả ra file mới
output_path = "Poland_no_dup.csv"
df_deduplicated.to_csv(output_path, index=False)


In [79]:
df_deduplicated.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1274 entries, 1130 to 899
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                1274 non-null   object 
 1   post_title        1274 non-null   object 
 2   post_content      1274 non-null   object 
 3   post_status       1274 non-null   object 
 4   post_author       1274 non-null   object 
 5   post_type         1274 non-null   object 
 6   post_date         1274 non-null   object 
 7   post_modified     1274 non-null   object 
 8   post_tags         1274 non-null   object 
 9   post_category     1274 non-null   object 
 10  default_category  1274 non-null   object 
 11  featured          1274 non-null   object 
 12  street            1274 non-null   object 
 13  street2           1274 non-null   object 
 14  city              1274 non-null   object 
 15  region            1274 non-null   object 
 16  country           1274 non-null   object 
 17

In [80]:
# Kiểm tra độ chính xác 
positive_keywords = [
    'vietnam', 'viet', 'việt', 'pho', 'bún', 'nem', 'saigon', "phở", 'sài gòn', 'hà nội', 'hanoi', 'bánh mì', 'bánh xèo',
    'halong', 'huế', 'bánh', 'goi cuon', 'bun cha', 'banh', 'thang', 'nam', 'sen', 'hoan kiem', 'wietnam', 'vietnamese',
    'vietnamese restaurant', 'tre', 'ha long', 'ha noi', 'sai gon', 'sajgon', 'hoang', 'ha-noi', 'com tam', 'hai', 'hoan', 'bami',
    'long', 'binh', 'banh mi', 'sao mai', 'song lam', 'con ga', 'phuong dong', 'linh'
]
negative_keywords = [
    'chinese', 'thai', 'japan', 'korean', 'fusion', 'asia', 'china','india', 'ramen', 'pasta', 'pizza', 'burger',
    'sushi', 'tapas', 'mexican', 'indian', 'kebab', 'italian', 'curry', "tai wan", "singapore", "malaysia", "korea",
    "hong kong", 'resort', 'hotel', 'pub', 'cafe', 'coffee', 'steak', 'park', 'inn', 'post', 'market', 'hall', 'bbq',
    'library', 'sandwich', 'cantonese'
]

# Bước 3: Tạo regex pattern
pattern_positive = re.compile('|'.join(positive_keywords), re.IGNORECASE)
pattern_negative = re.compile('|'.join(negative_keywords), re.IGNORECASE)

# Bước 4: Hàm gán tag
def tag_positive(text):
    if pd.isna(text):
        return ''
    return 'Vietnamese restaurant' if pattern_positive.search(text) else ''

def tag_negative(text):
    if pd.isna(text):
        return ''
    return 'Others' if pattern_negative.search(text) else ''

# Bước 5: Gán PositiveTag và NegativeTag
df_deduplicated['Pos'] = df_deduplicated.apply(
    lambda row: tag_positive(row['post_title']) or tag_positive(row['post_content']),
    axis=1
)

df_deduplicated['Neg'] = df_deduplicated.apply(
    lambda row: tag_negative(row['post_title']) or tag_negative(row['post_content']),
    axis=1
)

# Bước 6: Logic gán ReCheck?
def final_recheck_tag(row):
    if row['Pos'] != '' and row['Neg'] == '':
        return 'N'
    elif row['Pos'] == '' and row['Neg'] != '':
        return 'N'
    elif row['Pos'] == '' and row['Neg'] == '':
        return 'Y'
    else:
        return 'Y'

df_deduplicated['ReCheck?'] = df_deduplicated.apply(final_recheck_tag, axis=1)

In [81]:
# Tạo label mẫu
# Tạo 1 cột mới tên là rỗng mới là Y trong df_checkpoint
def assign_label(row):
    pos = row['Pos'] == 'Vietnamese restaurant'
    neg = row['Neg'] == 'Others'
    recheck = row['ReCheck?']

    if pos and not neg and recheck == 'N':
        return 1
    elif neg and not pos and recheck == 'N':
        return 0
    elif pos and neg and recheck == 'Y':
        return 0
    else:
        return ''

df_deduplicated['Y'] = df_deduplicated.apply(assign_label, axis=1)

df = df_deduplicated.copy()

# Xuất file để check manual
columns_to_export = [
    'post_title', 'post_content', 'website', 'google_profile', "Y", 'city'
]
df = df[columns_to_export]
df.to_csv('poland_labeled.csv', index=False)


In [82]:
df_deduplicated.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1274 entries, 1130 to 899
Data columns (total 34 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                1274 non-null   object 
 1   post_title        1274 non-null   object 
 2   post_content      1274 non-null   object 
 3   post_status       1274 non-null   object 
 4   post_author       1274 non-null   object 
 5   post_type         1274 non-null   object 
 6   post_date         1274 non-null   object 
 7   post_modified     1274 non-null   object 
 8   post_tags         1274 non-null   object 
 9   post_category     1274 non-null   object 
 10  default_category  1274 non-null   object 
 11  featured          1274 non-null   object 
 12  street            1274 non-null   object 
 13  street2           1274 non-null   object 
 14  city              1274 non-null   object 
 15  region            1274 non-null   object 
 16  country           1274 non-null   object 
 17

In [83]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1274 entries, 1130 to 899
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   post_title      1274 non-null   object
 1   post_content    1274 non-null   object
 2   website         480 non-null    object
 3   google_profile  1274 non-null   object
 4   Y               1274 non-null   object
 5   city            1274 non-null   object
dtypes: object(6)
memory usage: 69.7+ KB


In [84]:
# Đọc file labeled
df_labeled = pd.read_csv('poland_labeled.csv')
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_labeled['Y'].value_counts(dropna=False))

Y
1.0    597
0.0    343
NaN    334
Name: count, dtype: int64


In [85]:
print(df_deduplicated['Y'].value_counts(dropna=False))

Y
1    597
0    343
     334
Name: count, dtype: int64


In [86]:
# Kiểm tra lại nhanh số lượng giá trị thiếu (NaN) sau khi chuyển đổi
missing_summary = df_deduplicated.isna().sum()
missing_summary

ID                     0
post_title             0
post_content           0
post_status            0
post_author            0
post_type              0
post_date              0
post_modified          0
post_tags              0
post_category          0
default_category       0
featured               0
street                 0
street2                0
city                   0
region                 0
country                0
zip                    0
latitude               0
longitude              0
website              794
neighbourhood          0
facebook            1043
instagram           1260
twitter             1274
phone                 65
email                  0
logo                   0
google_profile         0
post_images            0
Pos                    0
Neg                    0
ReCheck?               0
Y                      0
dtype: int64

In [87]:
# Chuyển toàn bộ chuỗi rỗng hoặc chuỗi chỉ chứa khoảng trắng thành NaN
df_final = df_deduplicated.copy()

In [88]:
df_final.head(5)

,ID,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,...,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?,Y
1130,,"""Kim Loan"" Bar","""Kim Loan"" Bar located in Wymyślin 4a, 06-500 ...",publish,,,,,,",249,",...,NaN,+48735817957,,,https://maps.google.com/?q=place_id:ChIJ25lWvQ...,,,,Y,
181,,A Dong,"A DONG Niepołomice, Niepołomice. 2,249 likes ·...",publish,,,,,,",249,",...,NaN,+48797591763,,,https://maps.google.com/?q=place_id:ChIJJ7mf6I...,,,,Y,
430,,A Dong Quan,"(41) 368 21 94600 486 445Kielce, ul. Duża 5Tog...",publish,,,,,,",249,",...,NaN,+48413682194,,,https://maps.google.com/?q=place_id:ChIJ6xSY35...,,Vietnamese restaurant,,N,1
228,,A Dong Restaurant,"Restauracja Orientalna A Dong, Katowice. 1239 ...",publish,,,,,,",249,",...,NaN,+48322586662,,,https://maps.google.com/?q=place_id:ChIJd-ZHtk...,,,,Y,
829,,A chau,Skip to contentMenuStrona głównaGaleriaKarta d...,publish,,,,,,",249,",...,NaN,+48226652526,,,https://maps.google.com/?q=place_id:ChIJmcPHXy...,,Vietnamese restaurant,,N,1


In [89]:
# Gán giá trị mặc định nếu thiếu
df_final['post_type'] = df_final['post_type'].fillna('gd_place')
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
default_date = '2025-06-06 00:00:00'
df_final['post_date'] = df_final['post_date'].fillna(default_date)
df_final['post_modified'] = df_final['post_modified'].fillna(default_date)
df_final['post_author'] = df_final['post_author'].fillna('admin')
df_final["post_content"] = ""
df_final = df_final.replace(r'^\s*$', pd.NA, regex=True)
df_final.set_index('ID', inplace=True)
# Đổi tên cột id thành ID
df_final.head(5)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1274 entries, <NA> to <NA>
Data columns (total 33 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   post_title        1274 non-null   object 
 1   post_content      0 non-null      object 
 2   post_status       1274 non-null   object 
 3   post_author       0 non-null      object 
 4   post_type         0 non-null      object 
 5   post_date         0 non-null      object 
 6   post_modified     0 non-null      object 
 7   post_tags         0 non-null      object 
 8   post_category     1274 non-null   object 
 9   default_category  1274 non-null   object 
 10  featured          1274 non-null   object 
 11  street            1274 non-null   object 
 12  street2           0 non-null      object 
 13  city              1264 non-null   object 
 14  region            0 non-null      object 
 15  country           1274 non-null   object 
 16  zip               1264 non-null   object 
 1

In [90]:
df_final.head(5)
df_final.to_csv('poland_final.csv', index=False)

In [93]:
# đọc file labeled_new
df_labeled_new = pd.read_csv('poland_labeled_new.csv')
df_standardized= pd.read_csv('poland_final.csv')
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_labeled_new['Y'].value_counts(dropna=False))

Y
1.0    717
0.0    421
NaN    136
Name: count, dtype: int64


In [95]:
# Bổ sung giá trị cột Y từ df_labeled_new vào df_standardized dựa trên index (không có cột ID)
df_standardized['Y'] = df_labeled_new['Y'].values
# Lưu kết quả vào file CSV
output_path = 'poland_standardized_with_labels.csv'
# đọc poland_standardized_with_labels.csv
df_standardized.to_csv(output_path, index=False)
df_standardized_with_labels = pd.read_csv(output_path)  
# Kiểm tra số nhãn của từng unique giá trị trong cột 'Y'
print(df_standardized_with_labels['Y'].value_counts(dropna=False))
# Lưu lại file đã chuẩn hoá
df_standardized_with_labels.to_csv('poland_standardized_final.csv', index=False)
# in head 5 dòng
df_standardized_with_labels.head(5)

Y
1.0    717
0.0    421
NaN    136
Name: count, dtype: int64


,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?,Y
0,"""Kim Loan"" Bar",NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,4.873582e+10,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ25lWvQ...,NaN,NaN,NaN,Y,1.0
1,A Dong,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,4.879759e+10,NaN,NaN,https://maps.google.com/?q=place_id:ChIJJ7mf6I...,NaN,NaN,NaN,Y,1.0
2,A Dong Quan,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,4.841368e+10,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ6xSY35...,NaN,Vietnamese restaurant,NaN,N,1.0
3,A Dong Restaurant,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,4.832259e+10,NaN,NaN,https://maps.google.com/?q=place_id:ChIJd-ZHtk...,NaN,NaN,NaN,Y,1.0
4,A chau,NaN,publish,NaN,NaN,NaN,NaN,NaN,",249,",249,...,NaN,4.822665e+10,NaN,NaN,https://maps.google.com/?q=place_id:ChIJmcPHXy...,NaN,Vietnamese restaurant,NaN,N,1.0


In [96]:
# Gán giá trị mặc định nếu thiếu
df_standardized_with_labels['post_type'] = df_standardized_with_labels['post_type'].fillna('gd_place')
# Gán ngày mặc định nếu thiếu, đảm bảo đúng định dạng chuỗi
default_date = '2025-06-06 00:00:00'
df_standardized_with_labels['post_date'] = df_standardized_with_labels['post_date'].fillna(default_date)
df_standardized_with_labels['post_modified'] = df_standardized_with_labels['post_modified'].fillna(default_date)
df_standardized_with_labels['post_author'] = df_standardized_with_labels['post_author'].fillna('admin')
df_standardized_with_labels["post_content"] = ""
df_standardized_with_labels = df_standardized_with_labels.replace(r'^\s*$', pd.NA, regex=True)
# trích xuất file cuối chỉ có các dòng mà giá trị cột Y là 1 và xóa cột Y sau đó 
df_standardized_with_labels = df_standardized_with_labels[df_standardized_with_labels['Y'] == 1]
df_standardized_with_labels.drop(columns=['Y'], inplace=True)
df_standardized_with_labels.head(5)

,post_title,post_content,post_status,post_author,post_type,post_date,post_modified,post_tags,post_category,default_category,...,instagram,twitter,phone,email,logo,google_profile,post_images,Pos,Neg,ReCheck?
0,"""Kim Loan"" Bar",<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,4.873582e+10,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ25lWvQ...,NaN,NaN,NaN,Y
1,A Dong,<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,4.879759e+10,NaN,NaN,https://maps.google.com/?q=place_id:ChIJJ7mf6I...,NaN,NaN,NaN,Y
2,A Dong Quan,<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,4.841368e+10,NaN,NaN,https://maps.google.com/?q=place_id:ChIJ6xSY35...,NaN,Vietnamese restaurant,NaN,N
3,A Dong Restaurant,<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,4.832259e+10,NaN,NaN,https://maps.google.com/?q=place_id:ChIJd-ZHtk...,NaN,NaN,NaN,Y
4,A chau,<NA>,publish,admin,gd_place,2025-06-06 00:00:00,2025-06-06 00:00:00,NaN,",249,",249,...,NaN,NaN,4.822665e+10,NaN,NaN,https://maps.google.com/?q=place_id:ChIJmcPHXy...,NaN,Vietnamese restaurant,NaN,N


In [97]:
# So sánh định dạng từng cột của df_true với df_gd
def compare_column_formats(df1, df2):
    comparison = {}
    for col in df1.columns:
        if col in df2.columns:
            comparison[col] = {
                'df1_dtype': df1[col].dtype,
                'df2_dtype': df2[col].dtype,
                'df1_unique_count': df1[col].nunique(),
                'df2_unique_count': df2[col].nunique()
            }
        else:
            comparison[col] = {
                'df1_dtype': df1[col].dtype,
                'df2_dtype': None,
                'df1_unique_count': df1[col].nunique(),
                'df2_unique_count': None
            }
    return comparison
# So sánh định dạng cột của df_true với df_gd
comparison_result = compare_column_formats(df_standardized_with_labels, df_gd)
# In kết quả so sánh
for col, info in comparison_result.items():
    print(f"Cột: {col}")
    print(f"  - df_true dtype: {info['df1_dtype']}, unique count: {info['df1_unique_count']}")
    print(f"  - df_gd dtype: {info['df2_dtype']}, unique count: {info['df2_unique_count']}")
    print()
# Ép kiểu các cột trong df_true để phù hợp với df_gd
def convert_column_types(df, reference_df):
    for col in reference_df.columns:
        if col in df.columns:
            ref_dtype = reference_df[col].dtype
            if ref_dtype == 'object':
                df[col] = df[col].astype(str)
            elif ref_dtype == 'int64':
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
            elif ref_dtype == 'float64':
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0).astype(float)
            elif ref_dtype == 'datetime64[ns]':
                df[col] = pd.to_datetime(df[col], errors='coerce')
    return df   
# Chuyển đổi kiểu dữ liệu của df_true để phù hợp với df_gd
df_standardized_with_labels = convert_column_types(df_standardized_with_labels, df_gd)

Cột: post_title
  - df_true dtype: object, unique count: 610
  - df_gd dtype: object, unique count: 100

Cột: post_content
  - df_true dtype: object, unique count: 0
  - df_gd dtype: object, unique count: 19

Cột: post_status
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: post_author
  - df_true dtype: object, unique count: 1
  - df_gd dtype: int64, unique count: 19

Cột: post_type
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: post_date
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 100

Cột: post_modified
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 100

Cột: post_tags
  - df_true dtype: float64, unique count: 0
  - df_gd dtype: object, unique count: 1

Cột: post_category
  - df_true dtype: object, unique count: 1
  - df_gd dtype: object, unique count: 1

Cột: default_category
  - df_true dtype: int64, unique count: 1
  - df_gd 

In [98]:
df_standardized_with_labels.head(5)
# Xóa thông tin của post_title là Mama Pho, Pho & More và Van Binh
df_standardized_with_labels = df_standardized_with_labels[~df_standardized_with_labels['post_title'].isin(['Mama Pho, Pho & More', 'Van Binh'])]
# In ra số lượng dòng sau khi xóa
print(f"Số lượng dòng sau khi xóa: {len(df_standardized_with_labels)}")
# In ra thông tin của các cột
df_standardized_with_labels.info()
# Lưu lại file đã chuẩn hoá
df_standardized_with_labels.to_csv('poland_upload.csv', index=False)

Số lượng dòng sau khi xóa: 713
<class 'pandas.core.frame.DataFrame'>
Index: 713 entries, 0 to 1272
Data columns (total 32 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   post_title        713 non-null    object 
 1   post_content      713 non-null    object 
 2   post_status       713 non-null    object 
 3   post_author       713 non-null    int64  
 4   post_type         713 non-null    object 
 5   post_date         713 non-null    object 
 6   post_modified     713 non-null    object 
 7   post_tags         713 non-null    object 
 8   post_category     713 non-null    object 
 9   default_category  713 non-null    int64  
 10  featured          713 non-null    int64  
 11  street            713 non-null    object 
 12  street2           713 non-null    float64
 13  city              713 non-null    object 
 14  region            713 non-null    object 
 15  country           713 non-null    object 
 16  zip              

In [99]:
#Xóa cột neg, pos, recheck
df_standardized_with_labels.drop(columns=['Neg', 'Pos', 'ReCheck?'], inplace=True)
# Lưu lại file đã chuẩn hoá
df_standardized_with_labels.to_csv('poland_upload.csv', index=False)